# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muzammilsharf/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The Rule: A web page is flagged as high-risk for traffic decay if it receives a significant volume of visibility (impressions > 1000) but its actual Click-Through Rate (CTR) is severely below the median expected CTR for its current search ranking position. The risk score scales logarithmically with impression volume to prioritize high-traffic failures.

The Output Labels:
- Action Label: FLAG_FOR_REVIEW
- Reason Code: SEVERE_CTR_DEFICIT

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
import os
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download

print("Locating cached file...")
file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse", 
    repo_type="dataset", 
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print("Loading exact GSC columns from the drive...")
cols_to_load = ['gsc_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position']
df = pd.read_parquet(file_path, columns=cols_to_load)

# 1. Filter for valid data and impressions >= 1000
df_filtered = df[(df['gsc_data_available'] == True) & (df['gsc_impressions'] >= 1000)].copy()

# 2. Calculate the missing CTR column safely
df_filtered['ctr'] = df_filtered['gsc_clicks'] / df_filtered['gsc_impressions']

# 3. Round the average position to group into ranking buckets
df_filtered['position_bucket'] = df_filtered['gsc_avg_position'].round().astype(int)

# 4. Group by the position to find the baseline expected CTR
position_baselines = df_filtered.groupby('position_bucket')['ctr'].median().reset_index()
position_baselines.rename(columns={'ctr': 'expected_ctr'}, inplace=True)

# 5. Merge back and calculate the deficit score
df_scored = pd.merge(df_filtered, position_baselines, on='position_bucket', how='left')
df_scored['score'] = (df_scored['expected_ctr'] - df_scored['ctr']) * np.log1p(df_scored['gsc_impressions'])

# 6. Filter only pages with a deficit
df_scored = df_scored[df_scored['score'] > 0].copy()

# 7. Assign metadata and sort
df_scored['action_label'] = 'FLAG_FOR_REVIEW'
df_scored['reason_code'] = 'SEVERE_CTR_DEFICIT'
ranked_queue = df_scored.sort_values(by='score', ascending=False)

# 8. Export safely to the required directory
output_dir = '../outputs'
os.makedirs(output_dir, exist_ok=True)

output_cols = ['score', 'action_label', 'reason_code']
ranked_queue[output_cols].to_csv(f'{output_dir}/baseline_action_score.csv', index=False)

print("Queue written to baseline_action_score.csv successfully!")

Locating cached file...
Loading exact GSC columns from the drive...
Queue written to baseline_action_score.csv successfully!


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

1. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: Query is completely answered by a Google featured snippet (Zero-Click search).

2. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: The result is buried underneath an aggressive video or shopping carousel.

3. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: Purely navigational query where users only want a specific competitor's login page.

4. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: Ranking as an unclickable image-pack result rather than a standard text link.

5. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: Temporary paid ads are monopolizing the visible clicks for this specific keyword.

6. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: The query is localized, so global searchers see the result but do not click.

7. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: The meta title is heavily truncated on mobile devices, artificially depressing clicks.

8. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: Bot scraping traffic is artificially inflating the impression count without triggering clicks.

9. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: The page serves a highly technical audience that searches often but only clicks for specific version numbers.

10. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: The content is seasonal and currently outside of its core conversion window.

11. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: Users assume the site has a paywall based on past brand experience.

12. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: Ranking for an acronym that has multiple meanings, diluting the relevant audience.

13. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: The title tag includes a slightly outdated date (e.g., "2025 Guide"), deterring immediate clicks.

14. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: The URL is part of a multi-step tutorial series that users have already bookmarked.

15. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: Users are saving the URL directly to a read-it-later application.

16. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: The search snippet provides a direct phone number or address, solving the user's need instantly.

17. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: The ranking position recently improved, but the rolling CTR average hasn't caught up yet.

18. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: A highly controversial news topic where users rely only on trusted legacy news domains.

19. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: The query intent is transactional, but our page is strictly informational.

20. FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence — Wrong if: The deficit is a statistical anomaly caused by a single day of untargeted viral impressions

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks Analysis:
The baseline heavily flags position 2 and 3 pages with massive visibility (e.g., Row 433 with 39,003 impressions) but zero clicks. These are almost certainly "zero-click" queries, image packs, or direct snippet answers rather than genuine traffic decay. This rigid mathematical heuristic completely fails to contextualize SERP (Search Engine Results Page) features, which proves exactly why an ML model is required to weigh multiple signals simultaneously.

Leakage Check:
Confirmed. The target label trend_direction was entirely excluded from the environment. The logic strictly utilized gsc_impressions, gsc_avg_position, and calculated ctr. No future-window data or proxy labels leaked into the baseline generation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.